# 11. Categorical Data & Memory Optimization: Beginner Guide

### 📌 Overview
Master **11. Categorical Data & Memory Optimization: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Category Conversion**: Covers `df['col'] = df['col'].astype('category')`.
- **Ordered Categories**: Covers `pd.CategoricalDtype(ordered=True)`.
- **Deep Memory Inspection**: Covers `.memory_usage(deep=True)`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Categorical Encoding: `Series.astype('category')`
- **What it does:** Converts an object or integer Series into a memory-efficient Categorical representation using an integer code lookup table.
- **Syntax:** `Series.astype('category')`
- **Key Note:** For columns with repeated string labels (low cardinality), categorical encoding reduces RAM usage by up to 90% and accelerates groupby/sorting operations.
- **Dataset Application & Code Demonstration:** Applies Categorical Encoding on fintech records using columns `card_type`, `device_type`, `region`, `transaction_status` to demonstrate real-world execution.


In [2]:
df_optimized = df.copy()
cat_cols = ['card_type', 'transaction_status', 'device_type', 'region']
for col in cat_cols:
    df_optimized[col] = df_optimized[col].astype('category')
print('Categorical Encodings Head:\n', df_optimized[cat_cols].head(3))

Categorical Encodings Head:
   card_type transaction_status device_type region
0      Visa           Reversed      Mobile  North
1      Visa            Pending         POS   West
2      Visa          Completed      Mobile   East


### 🔹 Ordered Categories: `pd.CategoricalDtype()`
- **What it does:** Returns the data type object (dtype) describing the element type and byte order.
- **Syntax:** `ndarray.dtype / Series.dtype`
  - **Parameters:**
    - `object`: Target array or series.
- **Key Note:** Homogeneous data types ensure maximum SIMD CPU cache alignment.
- **Dataset Application & Code Demonstration:** Applies Ordered Categories on fintech records using columns `transaction_status` to demonstrate real-world execution.


In [3]:
status_order = pd.CategoricalDtype(categories=['pending', 'declined', 'approved'], ordered=True)
df_optimized['transaction_status'] = df_optimized['transaction_status'].astype(status_order)
print('Ordered Category Type Verified:', df_optimized['transaction_status'].dtype)

Ordered Category Type Verified: category


### 🔹 Deep Memory Profiling: `DataFrame.memory_usage()`
- **What it does:** Calculates the total memory footprint in bytes consumed by each column, traversing underlying Python object pointers when deep inspection is enabled.
- **Syntax:** `DataFrame.memory_usage()`
- **Key Note:** Shallow profiling (`deep=False`) only counts fixed pointer table sizes, underestimating object/string column memory by an order of magnitude.
- **Dataset Application & Code Demonstration:** Executes Deep Memory Profiling on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [4]:
raw_mem = df.memory_usage(deep=True).sum() / 1024
opt_mem = df_optimized.memory_usage(deep=True).sum() / 1024
print(f'Original Dataset Memory: {raw_mem:.1f} KB')
print(f'Optimized Dataset Memory: {opt_mem:.1f} KB ({(1 - opt_mem/raw_mem)*100:.1f}% RAM reduction)')

Original Dataset Memory: 6889.3 KB
Optimized Dataset Memory: 3737.5 KB (45.7% RAM reduction)


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: High-Throughput Memory Profiling Strategy
- **Objective:** Q1: High-Throughput Memory Profiling Strategy
- **Approach:** Write an automated optimization function that converts all low-cardinality string columns to categories and downcasts numerics.
- **Syntax:** `df[col].astype('category')` if cardinality < 50%

In [5]:
def optimize_dataframe(data):
    res = data.copy()
    for c in res.select_dtypes(include='object').columns:
        if res[c].nunique() / len(res) < 0.5:
            res[c] = res[c].astype('category')
    return res

opt_df = optimize_dataframe(df)
print('Automated Optimization Completed. Memory Saved:', (df.memory_usage(deep=True).sum() - opt_df.memory_usage(deep=True).sum()) // 1024, 'KB')

Automated Optimization Completed. Memory Saved: 4957 KB
